# Embeddings for Retrieval — AI Lab Instructor Notebook

*RAG & Retrieval · Hard*



7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
from collections import Counter

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

Setup

In [ ]:
import numpy as np
from collections import Counter

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Embeddings for Retrieval — Instructor Lab

Offline retrieval lab: build dense-ish and sparse embeddings, run top-k retrieval, evaluate Recall@k and MRR.

## Task 2: Section 0 — Corpus + labeled queries

## Section 0 — Corpus + labeled queries

In [ ]:
$10

In [ ]:
# Checks
check('docs_nonempty', len(DOCS) >= 6)
check('queries_nonempty', len(QUERIES) >= 3)

## Task 3: Section 1 — Tokenization + vocabulary

## Section 1 — Tokenization + vocabulary

In [ ]:
import re
_WORD_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?|\d+")

def tokenize(s: str):
    return _WORD_RE.findall(s.lower())

def build_vocab(texts, min_freq: int = 1):
    cnt = Counter()
    for t in texts:
        cnt.update(tokenize(t))
    vocab = [w for w, c in cnt.items() if c >= min_freq]
    vocab.sort()
    w2i = {w:i for i,w in enumerate(vocab)}
    return vocab, w2i, cnt

vocab, w2i, cnt = build_vocab(DOCS)
print('vocab_size', len(vocab))
check('vocab_nonempty', len(vocab) > 0)

In [ ]:
# Checks
check('vocab_nonempty', len(vocab) > 0)

## Task 4: Section 2 — Similarity metrics

## Section 2 — Similarity metrics

In [ ]:
def l2_normalize_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    nrm = np.sqrt(np.sum(X*X, axis=1, keepdims=True))
    return X / (nrm + eps)

def cosine_sim_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    # A: (n,d), B: (m,d) -> (n,m)
    A2 = l2_normalize_rows(A)
    B2 = l2_normalize_rows(B)
    return A2 @ B2.T

def top_k(sim_row: np.ndarray, k: int = 3):
    idx = np.argsort(-sim_row)[:k]
    return idx.tolist()

## Task 5: Section 3 — TF-IDF retrieval baseline

## Section 3 — TF-IDF retrieval baseline

In [ ]:
$11

## Task 6: Section 4 — Dense-ish baseline: co-occurrence + SVD word vectors

## Section 4 — Dense-ish baseline: co-occurrence + SVD word vectors

In [ ]:
$12

## Task 7: Section 5 — Evaluation: Recall@k and MRR

## Section 5 — Evaluation: Recall@k and MRR

In [ ]:
def recall_at_k(ranked, gold, k: int):
    return 1.0 if gold in ranked[:k] else 0.0

def mrr(ranked, gold):
    for i, doc_id in enumerate(ranked):
        if doc_id == gold:
            return 1.0 / (i + 1)
    return 0.0

def eval_retriever(fn, k: int = 3):
    r = []
    m = []
    for q, gold in QUERIES:
        ranked, _ = fn(q, k=k)
        r.append(recall_at_k(ranked, gold, k=k))
        m.append(mrr(ranked, gold))
    return float(np.mean(r)), float(np.mean(m))

r_tfidf, m_tfidf = eval_retriever(retrieve_tfidf, k=3)
r_dense, m_dense = eval_retriever(retrieve_dense, k=3)
print('TFIDF  Recall@3', r_tfidf, 'MRR', m_tfidf)
print('Dense  Recall@3', r_dense, 'MRR', m_dense)
check('metrics_finite', np.isfinite(r_tfidf) and np.isfinite(r_dense))

In [ ]:
# Checks
check('metrics_finite', np.isfinite(r_tfidf) and np.isfinite(r_dense))